# SCF and Multipole Expansions

galpy can represent arbitrary density distributions using basis-function expansions
(SCF) and multipole expansions. This notebook demonstrates both approaches.

In [ ]:
%matplotlib inline
import numpy
from matplotlib import pyplot as plt
from galpy import potential
from galpy.potential import (
    SCFPotential,
    MultipoleExpansionPotential,
    DiskSCFPotential,
    NFWPotential,
)
from galpy.orbit import Orbit

## SCFPotential from a density: prolate NFW

The `SCFPotential.from_density` class method builds an SCF expansion from any
density function. Let's use it to represent a prolate (flattened along R) NFW halo.

In [ ]:
# Define a prolate NFW density: stretch z by factor 1.5
nfw = NFWPotential(a=2.0, normalize=0.35)


def prolate_nfw_dens(R, z, phi=0.0):
    # Squeeze z to make it prolate
    return nfw.dens(R, z / 1.5) / 1.5


# Build SCF expansion
scf_prolate = SCFPotential.from_density(
    prolate_nfw_dens, N=10, L=6, a=2.0, symmetry="axi"
)

In [ ]:
# Compare densities along the z-axis
zs = numpy.linspace(0.01, 3.0, 100)
dens_true = [prolate_nfw_dens(0.01, z) for z in zs]
dens_scf = [scf_prolate.dens(0.01, z) for z in zs]

plt.semilogy(zs, dens_true, label="True density")
plt.semilogy(zs, dens_scf, "--", label="SCF expansion")
plt.xlabel(r"$z / R_0$")
plt.ylabel(r"$\rho$")
plt.legend()
plt.title("Prolate NFW density comparison")

In [ ]:
# Integrate an orbit in the prolate NFW SCF potential
o = Orbit([1.0, 0.1, 1.0, 0.0, 0.1, 0.0])
ts = numpy.linspace(0.0, 50.0, 10001)
o.integrate(ts, scf_prolate)
o.plot()

## MultipoleExpansionPotential from a density

`MultipoleExpansionPotential.from_density` uses a multipole expansion to
represent the potential of an arbitrary density distribution.

In [ ]:
# Build a multipole expansion of the same prolate NFW
mep = MultipoleExpansionPotential.from_density(
    prolate_nfw_dens, L=6, rgrid=numpy.geomspace(0.01, 50.0, 201), symmetry="axi"
)

In [ ]:
# Compare potentials
Rs = numpy.linspace(0.1, 3.0, 50)
pot_scf = [scf_prolate(R, 0.0) for R in Rs]
pot_mep = [mep(R, 0.0) for R in Rs]

plt.plot(Rs, pot_scf, label="SCF")
plt.plot(Rs, pot_mep, "--", label="Multipole")
plt.xlabel(r"$R / R_0$")
plt.ylabel(r"$\Phi$")
plt.legend()
plt.title("Potential comparison: SCF vs. Multipole")

## Computing SCF coefficients manually

You can also compute SCF coefficients yourself and pass them to `SCFPotential`:

In [ ]:
from galpy.potential import scf_compute_coeffs_axi

# Compute coefficients for the spherical NFW
Acos, Asin = scf_compute_coeffs_axi(lambda R, z: nfw.dens(R, z), N=10, L=2, a=2.0)

# Build SCF potential from coefficients
scf_manual = SCFPotential(Acos=Acos, Asin=Asin, a=2.0)
print("Manual SCF Phi(1,0):", scf_manual(1.0, 0.0))
print("Direct NFW Phi(1,0):", nfw(1.0, 0.0))

## DiskSCFPotential for disky potentials

`DiskSCFPotential` is designed for disk-like density distributions. It separates
the disk component from the spherical expansion to improve accuracy.

In [ ]:
# Double exponential disk: Sigma(R) * h(z)
# with Sigma(R) = exp(-3R), h(z) = exp(-27|z|)
dscf = DiskSCFPotential(
    dens=lambda R, z: 13.5 * numpy.exp(-3.0 * R) * numpy.exp(-27.0 * numpy.fabs(z)),
    Sigma={"type": "exp", "h": 1.0 / 3.0, "amp": 1.0},
    hz={"type": "exp", "h": 1.0 / 27.0},
    N=10,
    L=10,
    a=1.0,
)

# Plot the potential
dscf.plot(rmin=0.01, rmax=2.0, zmin=-0.3, zmax=0.3, nrs=50, nzs=50)

## Time-dependent MultipoleExpansionPotential

The `MultipoleExpansionPotential` supports time dependence. For example,
we can represent a slowly rotating bar perturbation.

In [ ]:
# Time-dependent density: a rotating triaxial perturbation
def rotating_bar_dens(R, z, phi=0.0, t=0.0):
    OmegaP = 1.5
    phi_bar = phi - OmegaP * t
    r2 = R**2 + z**2
    # Hernquist-like density with cos(2*phi) perturbation
    rho0 = 1.0 / (r2**0.5 * (1 + r2**0.5) ** 3)
    return rho0 * (1.0 + 0.3 * numpy.cos(2 * phi_bar))


mep_td = MultipoleExpansionPotential.from_density(
    rotating_bar_dens,
    L=4,
    rgrid=numpy.geomspace(0.01, 20.0, 101),
    symmetry=None,
    tgrid=numpy.linspace(0.0, 4.0 * numpy.pi / 1.5, 51),
)

# Potential at fixed R, z as function of phi at two times
phis = numpy.linspace(0, 2 * numpy.pi, 100)
for t in [0.0, 1.0]:
    vals = [mep_td(1.0, 0.0, phi=phi, t=t) for phi in phis]
    plt.plot(phis, vals, label=f"t = {t:.1f}")
plt.xlabel(r"$\phi$")
plt.ylabel(r"$\Phi$")
plt.legend()
plt.title("Time-dependent multipole expansion")